# SmartRoad - Traffic Accident Analytics

Computational Analytics Assignment 2. This notebook performs acquisition, cleaning, EDA, visualization, feature engineering, classification, cross-validation, evaluation and hotspot clustering.

**Academic data note:** the project includes a synthetic DEMO dataset for local validation. For submission, download the official 2025 DfT collisions CSV using `download_data.py`.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from analytics import load_data, preprocess, clean_numeric_ranges, build_model, permutation_importance_table, fit_cluster, make_charts


In [ ]:
DATA = PROJECT_ROOT / 'data' / 'road_collisions_2025.csv'
if not DATA.exists():
    DATA = PROJECT_ROOT / 'data' / 'demo_accidents.csv'
print('Using:', DATA)
raw = load_data(DATA)
print(raw.shape)
display(raw.head())


## 1. Data quality and preprocessing

In [ ]:
print('Duplicates:', raw.duplicated().sum())
print('Missing cells:', int(raw.isna().sum().sum()))

df = clean_numeric_ranges(preprocess(raw))
print('Rows after duplicate removal:', len(df))
display(df[['date','hour','month','day_name','weekend','night','rush_hour','severity_label','high_severity']].head())

## 2. Exploratory analysis

In [ ]:
display(df.describe(include='all').T.head(25))
display(df['severity_label'].value_counts().rename_axis('severity').to_frame('count'))

## 3. Required visualizations (minimum 5 chart types)

In [ ]:
chart_names = make_charts(df, str(PROJECT_ROOT / 'outputs' / 'notebook_charts'))
chart_names


The generated chart set includes a line chart, bar chart, histogram, heatmap, boxplot and geographic scatter plot.

## 4. Predictive modeling

In [ ]:
rf, X_test, y_test, pred, metrics, features = build_model(df, 'Random Forest')
print(metrics)
print('Features:', features)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.title('Random Forest Confusion Matrix')
plt.show()

In [ ]:
imp = permutation_importance_table(rf, X_test, y_test)
display(imp.head(15))

## 5. Hotspot clustering

In [ ]:
cluster_points, cluster_summary = fit_cluster(df, 8)
display(cluster_summary.sort_values('high_severity_rate', ascending=False))

## 6. Conclusions

Use the output metrics, charts and hotspot summary generated from the official dataset for the final submission. The included demo run is only for software validation and should not be presented as official accident statistics.